# Dragon Distributed Dictionary (`DDict`) Tutorial

**Estimated time:** ~20 minutes  
**Format:** 5 short exercises — each has a problem and a hidden answer you can reveal.

---

## What is `DDict`?

A `DDict` is Dragon's **distributed in-memory key-value store**. It looks and
feels like a Python `dict`, but the data lives across **one or more manager
processes** that can span multiple nodes. The best part is that the organization
of the data is transparent to the user. When your code interacts with it, the
code does not need to know the location of any key/value pair stored in it.

When you create a DDict you specify some number of managers and nodes on which
the DDict will run. A DDict automatically starts up a specified number of
managers and an orchestrator (not shown in the picture) and your client code
interacts with these processes to transparently provide you with a distributed
key/value store that can scale to any size and automatically distributes your
data evenly across all the mangers. But, your code sees it as just one dictionary
available from any client that has a handle to it. One client process creates the
DDict while other client processes can be passed the DDict so they
too can use it. 


<img src="images/ddict_overview.png" width="80%">

Key characteristics:
- Standard `dict`-like interface (`d[key] = val`, `d[key]`, `del d[key]`, `len(d)`, ...)
- Values are transparently serialized with `cloudpickle` or a pickler of your choice
- Data is shared safely across **multiple processes** without locks
- Supports **checkpointing**, **persistent keys**, and **filtering** 
- Other functionality not covered here like **broadcast puts** and **batch puts**.
- Data is automatically balanced across all the managers

---

## Setup — run this first!

You should already have already *pip* installed Dragon in a Python environment.
Then run `dragon-jupyter` from within the environment and open the printed URL in
your web browswer. Once you have Jupyter running, then open this jupyter notebook
to begin and run the next cell.

In [ ]:
import dragon
import multiprocessing as mp

# This is needed when running Dragon. You import dragon (as on the first line above) and then
# you set the multiprocessing start method to "dragon".
mp.set_start_method("dragon")

from dragon.data.ddict import DDict
print("Dragon ready ✓")

Running the cell above loads Dragon and sets multiprocessing to use the *dragon* start method to start new processes and handle all the interprocess communication and synchronization. 

---

## Exercise 1 — Create, write, read, and destroy a `DDict`

**Background:**

The DDict class contains the client code needed for starting and interacting with a Dragon Distributed Dictionary. Here is how to create a DDict and start it up.

```python
d = DDict(managers_per_node, n_nodes, total_mem_bytes)
```

This tells Dragon to start a DDict with `managers_per_node * n_nodes` managers. The total_mem_bytes is spread across all the managers. For more information on creating DDicts [see the DDict API](https://dragonhpc.github.io/dragon/doc/_build/html/ref/data/dragon.data.DDict.html#dragon.data.DDict). 

- `managers_per_node=1`, `n_nodes=1` → one manager, single node
- `total_mem=64*1024*1024` → 64 MB pool
- Always call `d.destroy()` when finished

**Your task:**

1. Create a `DDict` with 1 manager, 1 node, 64 MB
2. Store `d["hello"] = "world"` and `d["answer"] = 42`
3. Read both values back and print them
4. Print the number of items (`len`)
5. Destroy the dictionary

In [ ]:
# ── Exercise 1 ── your code here ─────────────────────────────────────────────


<details>
<summary><b>▶ Show Answer</b></summary>

```python
d = DDict(1, 1, 64 * 1024 * 1024)

d["hello"] = "world"
d["answer"] = 42

print(d["hello"])      # world
print(d["answer"])     # 42
print(len(d))          # 2

d.destroy()
```

</details>

---

## Exercise 2 — Dict operations: iterate, check membership, delete

**Background:**

`DDict` supports the standard dict (i.e. mapping) interface of Python. Many more operations are available too. See the [DDict API documentation](https://dragonhpc.github.io/dragon/doc/_build/html/ref/data/dragon.data.DDict.html#dragon-data-ddict) for more details.

| Operation | Syntax |
|---|---|
| Membership | `key in d` |
| Default get | `d.get(key, default)` |
| Delete | `del d[key]` |
| Keys | `d.keys()` |
| Values | `d.values()` |
| Items | `d.items()` |

**Your task:**

1. Create a `DDict` (1 manager, 1 node, 32 MB)
2. Store `{"cat": 1, "dog": 2, "bird": 3}`
3. Print all keys by iterating over `d.keys()`
4. Check whether `"dog"` and `"fish"` are in the dictionary
5. Delete `"cat"` and confirm the length dropped to 2
6. Use `d.get("fish", "not found")` and print the result
7. Destroy the dictionary

In [ ]:
# ── Exercise 2 ── your code here ─────────────────────────────────────────────


<details>
<summary><b>▶ Show Answer</b></summary>

```python
d = DDict(1, 1, 32 * 1024 * 1024)

d["cat"]  = 1
d["dog"]  = 2
d["bird"] = 3

print("keys:", list(d.keys()))          # ['cat', 'dog', 'bird'] (order may vary)
print("dog in d:", "dog" in d)          # True
print("fish in d:", "fish" in d)        # False

del d["cat"]
print("len after delete:", len(d))      # 2

print(d.get("fish", "not found"))       # not found

d.destroy()
```

</details>

---

## Exercise 3 — Sharing a `DDict` across processes

**Background:**

A `DDict` is serializable. Serializable resources can be passed between processes and attached in the receiving process. Python does this automatically for you when you pass a serializable object as an argument in multiprocessing. So you can pass a DDict as an argument to a `Process` (a newly created process) and the child will share the **same** DDict as the parent. 

**NOTE:**

**When it comes to Jupyter, Python's multiprocessing Process does not work as you'd like it when a process' target is a function defined in a notebook. This is a known incompatibility between Jupyter and Python and it would be an inconvenience here except that Dragon has its own native version of Process that can be used in place of multiprocessing's Process. [Dragon Native Process](https://dragonhpc.github.io/dragon/doc/_build/html/ref/native/dragon.native.process.html#dragon.native.process.Process) objects can be passed arguments like a DDict in the same way as a [multiprocessing Process](https://docs.python.org/3/library/multiprocessing.html#multiprocessing.Process) and Dragon Native Proccesses do not have the same Jupyter notebook incompatability. So for convenience, we'll use Dragon's Native Process when creating new processes in this notebook.**

```python
def worker(d, key, value):
    d[key] = value   # writes into the shared DDict
    d["hello"] = "world" # keys and values don't have to be passed to child proceses

proc = dn.Process(target=worker, args=(d, "x", 99)) # creates a Dragon Native Process
proc.start() # starts the process
proc.join()  # waits for the process to exit
print(d["hello"]) # and then prints world
```

**Your task:**

Write a function `count_words(d, text)` that splits `text` on spaces and stores
`d[word] = count` for every unique word in the text. To make this a bit easier, try
using the `Counter` class from the [collections module](https://docs.python.org/3/library/collections.html#counter-objects). The `Counter` class, given a 
list of words, will construct a dictionary which maps each word to the frequency it 
appeared in the list.

Calling `s.split()` on a `str` object `s` in Python will return a list of all the space 
separated *words* of the string. 

Then:
1. Create a `DDict` (2 managers, 1 node, 64 MB)
2. Launch **3 processes**, each calling `count_words` with a different sentence
3. After all processes finish, print every key-value pair in the dictionary
4. Destroy the dictionary

Sentences to use:
```
"the quick brown fox"
"the fox jumped over the fence"
"a quick brown cat"
```

In [ ]:
# ── Exercise 3 ── your code here ─────────────────────────────────────────────

from collections import Counter
import dragon.native as dn

sentences = [
    "the quick brown fox",
    "the fox jumped over the fence",
    "a quick brown cat",
]

def count_words(d, text):
    pass

<details>
<summary><b>▶ Show Answer</b></summary>

```python
from collections import Counter

sentences = [
    "the quick brown fox",
    "the fox jumped over the fence",
    "a quick brown cat",
]

def count_words(d, text):
    for word, count in Counter(text.split()).items():
        d[word] = count

d = DDict(2, 1, 64 * 1024 * 1024)

procs = [dn.Process(target=count_words, args=(d, s)) for s in sentences]
for p in procs:
    p.start()
for p in procs:
    p.join()

for k, v in d.items():
    print(f"  {k}: {v}")

d.destroy()
```

**Getting Correct Word Counts in Parallel** 

Because multiple processes write the **same** words (e.g. `"the"`, `"fox"`),
whichever process writes last wins. If you want correct word counts across all processes, you can use the atomic update `fetch_add` to add the occurrences of each word.
The `fetch_add` will operate atomically to add an integer value to a specified key/value pair. 

```python
def count_words(d, text):
    for word, count in Counter(text.split()).items():
        d.fetch_add(word, count)
```

If you replace your `count_words` with the one above you will see a decoding error when trying to retrieve the value from the DDict. That's because a `fetch_add` value is encoded so `fetch_add` can be used by both Python and C++. To retrieve a `fetch_add` value from the DDict, use `fetch_add(key, 0)` to get it and leave it unchanged. Using `fetch_add` to retrieve fetch/added values works in both Python and C++. So instead of writing the `for k, v in d.items()` to print the values, you can write this instead.

```python
for k in d:
    print(f"  {k}: {d.fetch_add(k,0)}")
```
</details>

---

## Exercise 4 — Persistent keys and `wait_for_keys`

**Background:**

By default, every key in a `DDict` is *checkpoint-scoped* — it belongs to one checkpoint
and will be retired when that checkpoint leaves the working set.

A **persistent key** (`d.pput(key, value)`) is stored independently of checkpoints.
It survives across all checkpoint advances and is never retired automatically.

`wait_for_keys=True` (requires `working_set_size >= 2`) makes any read of a
checkpoint-scoped key **block** until another process writes it, removing the need
for manual synchronization between reading and writing processes.

```python
d = DDict(1, 1, 64*1024*1024, wait_for_keys=True, working_set_size=2)
d.pput("config", {"lr": 0.01, "epochs": 10})   # persistent

def trainer(d):
    cfg = d["config"]           # always available
    d["result"] = cfg["lr"] * 2

proc = dn.Process(target=trainer, args=(d,))
proc.start()
print(d["result"])              # 0.02 - blocks until is is available
d.destroy()
proc.join()
```

**Your task:**

1. Create a `DDict` with `wait_for_keys=True, working_set_size=2`
2. Persistently store a shared config under the key `"config"`
   with `{"scale": 10, "offset": 5}`
3. Launch 4 worker processes. Each worker should:
   - Read `"config"` from the DDict
   - Write `d[f"result_{worker_id}"] = worker_id * scale + offset`
4. In the main process, after all workers finish, print all `result_*` values
5. Destroy the dictionary

In [ ]:
# ── Exercise 4 ── your code here ─────────────────────────────────────────────

def worker_4(d, worker_id):
    pass  # replace with your implementation

<details>
<summary><b>▶ Show Answer</b></summary>

```python
def worker_4(d, worker_id):
    cfg = d["config"]
    result = worker_id * cfg["scale"] + cfg["offset"]
    d[f"result_{worker_id}"] = result

d = DDict(1, 1, 64 * 1024 * 1024, wait_for_keys=True, working_set_size=2)
d.pput("config", {"scale": 10, "offset": 5})

procs = [dn.Process(target=worker_4, args=(d, i)) for i in range(4)]
for p in procs:
    p.start()

for i in range(4):
    print(f"result_{i} = {d[f'result_{i}']}")

d.destroy()

for p in procs:
    p.join()
```

Expected output:
```
result_0 = 5
result_1 = 15
result_2 = 25
result_3 = 35
```

</details>

---

## Exercise 5 — Scalably filtering a `DDict`

**Background:**

`d.filter` can return anything a user would like by distributing a function to run locally with
each manager of the DDict. The user gets to specify the code to be run. The user's function is passed
a copy of the DDict that targets just one manager on the same node where the function runs - communication
between the function and the manager-directed DDict is all on-node to be efficient as possible. The function is 
also passed a Queue to which the results of filtering should be written. Optionally, the user's filter function 
can be passed additional arguments specified when calling the filter function. Consider this code that filters out 
all keys that don't start with the given string.

```python
def filter_keys(local_d, out_queue, startswith_str):
    for key in local_d:
        try:
            if k.startswith(startswith_str):
                out_queue.put(key)
        except:
            pass 
```

The `local_d` is a DDict that targets only the local manager that the function will run 
against. The filter function will invoke this function in parallel for each manager of the 
DDict and will run it locally, on the same node as the manager with which it will work.

The filter function let's you combine these results from different managers however you would 
like them combined. The filter function automatically combines the results using a tree structure 
to allow filter to scale to the limits of any machine. Let's say we want to get the keys in 
alphabetical order. Then we write a comparator that returns `True` when x is less than y as in:

```python
def filter_keys_comparator(x, y):
    return x < y
```

Finally, we can invoke the filter function on the DDict as follows.

```python
d = DDict(2, 1, 64*1024*1024)
d["apple"] = 5
d["apricot"] = 3
d["banana"] = 7
d["blueberry"] = 2

# Filter for keys starting with 'a'.
with d.filter(filter_keys, ("a",), filter_keys_comparator) as candidates:
    for candidate in candidates:
        candidate_list.append(candidate) # yields: apple, apricot
```

**Your task:**

1. Create a `DDict` (2 managers, 1 node, 64 MB)
2. Populate it with sensor data: `{"temp_room1": 72.5, "temp_room2": 75.0, "humidity_room1": 45, "humidity_room2": 50}`
3. Use `d.filter` to extract only keys starting with `"temp_"` and a temperature >= 74 degrees.
4. Print all temperature readings from the filtered result
5. Use another filter to find all humidity values >= 45
6. Destroy the dictionary

In [ ]:
# ── Exercise 5 ── your code here ─────────────────────────────────────────────


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def filter_readings(local_d, out_queue, startswith_str, exceeds_val):
    for key in local_d:
        try:
            if key.startswith(startswith_str):
                value = local_d[key]
                if value >= exceeds_val:
                    out_queue.put((key, value))
        except Exception as ex:
            print(ex)

def filter_comparator(x,y):
    return True # We don't care about order.

d = DDict(1, 1, 64 * 1024 * 1024)

# Populate with sensor data
d["temp_room1"] = 72.5
d["temp_room2"] = 75.0
d["humidity_room1"] = 45
d["humidity_room2"] = 50

# Filter for temperature readings
print("Temperature readings (>= 74):")
with d.filter(filter_readings, ("temp", 74), filter_comparator) as candidates:
    for key, value in candidates:
        print(f"  {key}: {value}")

# Filter for humidity values >= 45
print("\nHigh humidity readings (>= 45):")
with d.filter(filter_readings, ("humidity", 45), filter_comparator) as candidates:
    for key, value in candidates:
        print(f"  {key}: {value}")

d.destroy()
```

**Expected output:**

```
Temperature readings (>= 74):
  temp_room2: 75.0

High readings (>= 45):
  humidity_room1: 45
  humidity_room2: 50
```

</details>

---

## Summary

This tutorial presented examples and exercises to demonstrate that the `DDict` can be used like the built-in `dict` type in Python. While a Python `dict` is limited
to one node, the Dragon `DDict` can span all the nodes of your supercomputing allocation while still able to run on a laptop - just at smaller scale. The `DDict` automatically 
distributes your data evenly over all the nodes and managers on which it is configured to run.  

The `DDict` supports concepts that are unique to it as well. It supports checkpointing while not requiring explicit quiescing of computation by introducing the concept of a 
working set of checkpoints. It provides an atomic counter via the `fetch_add` method. It provides a scalable `filter` method that can be used to distribute compute to
your data and deliver results reliably at any scale.

Key concepts introduced in this tutorial are summarized below. 

| Concept | API |
|---|---|
| Create | `DDict(managers_per_node, n_nodes, total_bytes)` |
| Write / Read | `d[key] = val` / `d[key]` |
| Delete | `del d[key]` |
| Size | `len(d)` |
| Iterate | `d.keys()`, `d.values()`, `d.items()` |
| Membership | `key in d` |
| Default read | `d.get(key, default)` |
| Persistent write | `d.pput(key, val)` |
| Filter | `d.filter(predicate)` |
| Synchronize | `wait_for_keys=True` + `working_set_size >= 2` |
| Atomic fetch_add | `d.fetch_add(key, val)` |
| Cleanup | `d.destroy()` |

---

### What's next?

More information can be found in the [DDict Documentation](https://dragonhpc.github.io/dragon/doc/_build/html/ref/data/dragon.data.DDict.html#dragon.data.DDict). 
Some of the features not introduced in this tutorial but worth checking out are:

- **Checkpointing:** `d.checkpoint()`, `d.rollback()` — snapshot the DDict state across training iterations or simulation steps
- **Broadcast put:** `d.bput(key, val)` — replicate a value to every manager for node-local access
- **Custom picklers:** `with d.pickler(key_pickler, value_pickler)` — checkout XPickler for use with the C++ client for the `DDict`.
- **Multi-node:** change `n_nodes` and `managers_per_node` to scale across your allocation